# Pruebas del agente Secretario AMPA V3.5

Notebook único para probar componentes de forma aislada.

Las pruebas externas están desactivadas inicialmente.

In [1]:
from pathlib import Path
import json
import os
import sqlite3
import sys

RUTA_ACTUAL = Path.cwd()

if RUTA_ACTUAL.name == "notebooks":
    RAIZ_PROYECTO = RUTA_ACTUAL.parent
else:
    RAIZ_PROYECTO = RUTA_ACTUAL

if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

os.chdir(RAIZ_PROYECTO)

print("Raíz:", RAIZ_PROYECTO)
print("Python:", sys.version.split()[0])

Raíz: c:\Users\Dario\Documents\Caperta_Botcamp\Apuntes_Juan\5-Data_Engineering\2-IA_Generativa\Proyecto_IA_gen\Secretario_AMPA
Python: 3.12.10


## 1. Configuración, prompts, tools y funciones

In [2]:
from src.parametros import (
    DATABASE_PATH,
    RAG_PATH,
    LLM_MODEL,
    TIMEZONE,
)
from src.prompts import cargar_prompts
from src.tools import tools
from src.funciones import funciones

prompts = cargar_prompts()

print("Modelo:", LLM_MODEL)
print("Zona horaria:", TIMEZONE)
print("Base de datos:", DATABASE_PATH)
print("RAG:", RAG_PATH)
print("Prompts:", list(prompts))
print("Tools:", list(tools))
print("Funciones:", list(funciones))

Modelo: llama-3.1-8b-instant
Zona horaria: Europe/Madrid
Base de datos: data\secretario_ampa.db
RAG: data\rag\correos_historicos.jsonl
Prompts: ['reglas', 'clasificacion', 'redaccion', 'reunion', 'whatsapp']
Tools: ['obtener_correos_no_leidos', 'clasificar_correo', 'consultar_antecedentes_gmail', 'redactar_borrador', 'crear_borrador', 'marcar_como_leido', 'extraer_reunion', 'consultar_disponibilidad', 'crear_resumen_whatsapp', 'enviar_whatsapp', 'registrar_correo']
Funciones: ['obtener_correos_no_leidos', 'clasificar_correo', 'consultar_antecedentes_gmail', 'redactar_borrador', 'crear_borrador', 'marcar_como_leido', 'extraer_reunion', 'consultar_disponibilidad', 'crear_resumen_whatsapp', 'enviar_whatsapp', 'registrar_correo']


## 2. Clasificación con correo simulado

In [3]:
EJECUTAR_CLASIFICACION = False

correo_prueba = {
    "remitente": "familia@example.com",
    "asunto": "Consulta sobre inscripción",
    "cuerpo": "¿Qué documentación necesito para inscribirme?",
}

if EJECUTAR_CLASIFICACION:
    resultado = funciones["clasificar_correo"](
        correo_prueba,
        prompts,
    )
    print(json.dumps(resultado, indent=2, ensure_ascii=False))
else:
    print("Prueba desactivada.")

Prueba desactivada.


## 3. Lectura de Gmail sin modificar mensajes

In [4]:
EJECUTAR_GMAIL = False

if EJECUTAR_GMAIL:
    correos = funciones["obtener_correos_no_leidos"]()

    print("Correos:", len(correos))

    for correo in correos:
        print(correo.get("fecha"), correo.get("asunto"))
else:
    print("Prueba desactivada.")

Prueba desactivada.


## 4. Consulta del RAG local

In [3]:
from src.rag import (
    cargar_documentos,
    consultar_antecedentes_gmail,
)

documentos = cargar_documentos()

print("Documentos:", len(documentos))
print(
    consultar_antecedentes_gmail(
        "inscripción, documentación y cuota"
    )
)

Documentos: 1
No se encontraron correos anteriores relevantes para esta consulta.


In [4]:
from src.rag import cargar_documentos

documentos = cargar_documentos()

print("Documentos cargados:", len(documentos))

for documento in documentos:
    print("\nAsunto:", documento.get("asunto"))
    print("Destinatario:", documento.get("destinatario"))
    print("Cuerpo:", documento.get("cuerpo"))

Documentos cargados: 1

Asunto: Solicitud de reunión con el AMPA
Destinatario: guaehtc@gmail.com
Cuerpo: Buenos días:

Gracias por vuestra solicitud de reunión. Hemos comprobado el calendario y
tenemos disponibilidad el martes 2026-06-30 a las 17:30 o miércoles
2026-07-01 a las 17:30.

¿Os viene bien alguna de estas opciones?

Un saludo,
Secretaría del AMPA


## 5. Redacción sin crear borrador en Gmail

In [1]:
EJECUTAR_REDACCION = False

if EJECUTAR_REDACCION:
    contexto = funciones[
        "consultar_antecedentes_gmail"
    ](
        correo_prueba["asunto"]
        + " "
        + correo_prueba["cuerpo"]
    )

    borrador = funciones["redactar_borrador"](
        correo=correo_prueba,
        prompts=prompts,
        tipo="respuesta",
        contexto_rag=contexto,
    )

    print(json.dumps(borrador, indent=2, ensure_ascii=False))
else:
    print("Prueba desactivada.")

Prueba desactivada.


## 6. Extracción y consulta de reunión

In [6]:
from src.calendar import parsear_fecha, normalizar_opciones

print(
    parsear_fecha(
        "8 de julio de 2026"
    )
)

print(
    normalizar_opciones([
        {
            "dia": "miércoles",
            "fecha": "8 de julio de 2026",
            "hora": "17:30",
        },
        {
            "dia": "jueves",
            "fecha": "9 de julio de 2026",
            "hora": "18:00",
        },
    ])
)

2026-07-08
[{'dia': 'miércoles', 'fecha': '2026-07-08', 'hora': '17:30'}, {'dia': 'jueves', 'fecha': '2026-07-09', 'hora': '18:00'}]


In [7]:
EJECUTAR_REUNION = True

correo_reunion = {
    "remitente": "familia@example.com",
    "asunto": "Solicitud de reunión con el AMPA",
    "cuerpo": (
        "Buenos días. Nos gustaría reunirnos con el AMPA. "
        "Podemos el miércoles 8 de julio de 2026 a las 17:30 "
        "o el jueves 9 de julio de 2026 a las 18:00. "
        "La reunión duraría aproximadamente una hora."
    ),
}

reunion = funciones["extraer_reunion"](
    correo_reunion,
    prompts,
)

print("Extracción:")
print(
    json.dumps(
        reunion,
        indent=2,
        ensure_ascii=False,
    )
)

disponibilidad = funciones[
    "consultar_disponibilidad"
](
    reunion.get("opciones", []),
    reunion.get("duracion_minutos", 60),
)

print("\nCalendar:")
print(
    json.dumps(
        disponibilidad,
        indent=2,
        ensure_ascii=False,
    )
)

Extracción:
{
  "tipo": "solicitud",
  "motivo": "Solicitud de reunión con el AMPA",
  "duracion_minutos": 60,
  "opciones": [
    {
      "dia": "miércoles",
      "fecha": "2026-07-08",
      "hora": "17:30"
    },
    {
      "dia": "jueves",
      "fecha": "2026-07-09",
      "hora": "18:00"
    }
  ]
}

Calendar:
{
  "ok": true,
  "estado": "disponibilidad_consultada",
  "opciones": [
    {
      "dia": "miércoles",
      "fecha": "2026-07-08",
      "hora": "17:30",
      "hora_hasta": "18:30",
      "inicio_iso": "2026-07-08T17:30:00+02:00",
      "disponible": false
    },
    {
      "dia": "jueves",
      "fecha": "2026-07-09",
      "hora": "18:00",
      "hora_hasta": "19:00",
      "inicio_iso": "2026-07-09T18:00:00+02:00",
      "disponible": false
    }
  ]
}


## 7. WhatsApp simulado

In [ ]:
EJECUTAR_WHATSAPP = False

correo_urgente = {
    "message_id": "prueba_notebook_001",
    "asunto": "Riesgo en el patio",
    "cuerpo": "Hay una pieza metálica suelta y puede causar cortes.",
}

if EJECUTAR_WHATSAPP:
    alerta = funciones[
        "crear_resumen_whatsapp"
    ](
        correo_urgente,
        prompts,
    )

    resultado = funciones["enviar_whatsapp"](
        message_id=correo_urgente["message_id"],
        resumen=alerta["resumen"],
        tipo_riesgo=alerta["tipo_riesgo"],
    )

    print(json.dumps(resultado, indent=2, ensure_ascii=False))
else:
    print("Prueba desactivada.")

## 8. Memoria SQLite

In [ ]:
from src.memoria import (
    inicializar_memoria,
    obtener_pendientes_revision,
)

inicializar_memoria()

print("Pendientes de revisión:")
for fila in obtener_pendientes_revision():
    print(fila)

## 9. Ejecución completa

In [ ]:
from main import ejecutar_ciclo

EJECUTAR_AGENTE = False

if EJECUTAR_AGENTE:
    print(
        json.dumps(
            ejecutar_ciclo(),
            indent=2,
            ensure_ascii=False,
        )
    )
else:
    print("Ejecución completa desactivada.")